# Open Notebook & Additional Resources

<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ORM_AI_Agents_Bootcamp/blob/main/demo/DAY_2_DEMO_Session_3_langgraph_planner_thinking_vs_unified_deep_research.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://learning.oreilly.com/library/view/ai-agents-the/0642572247775/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="AI Agents Book – Read on O'Reilly"/>
</a>





# Demo: Planner + reasoning vs unified ReAct (deep-research style)

This notebook compares two **LangGraph** architectures on the same **real** research task—**no mocked APIs**.

## Architecture A — staged pipeline (more explicit graph)

1. **Plan** — one call to a **planner model** on [OpenRouter](https://openrouter.ai) with an optional **reasoning budget** (`extra_body.reasoning`). The planner emits a structured plan: sub-questions, Exa queries, and a section outline.
2. **Gather** — deterministic code runs **Exa** (`search_and_contents`) for each planned query and concatenates snippets (no LLM).
3. **Synthesize** — a separate **writer** model (no reasoning flags in this demo) turns findings into a short cited report.

Flow: `START → plan → gather → synthesize → END`.

## Architecture B — unified ReAct agent (simpler graph)

A single **`create_react_agent`** loop: one chat model decides when to call **`exa_web_search`** and when to answer. By default this path uses **no** OpenRouter `reasoning` block so it contrasts cleanly with the planner; you can optionally turn reasoning on for **every** model step via env vars (see below)—useful to feel latency and cost when reasoning runs **per tool turn**.

## Reasoning budget: why it matters (OpenRouter)

OpenRouter exposes a unified **`reasoning`** object on chat requests. Official guide: **[Reasoning tokens — best practices](https://openrouter.ai/docs/guides/best-practices/reasoning-tokens)**.

**Why tune it in agent designs?**

- **Cost** — Reasoning tokens count as **output tokens** and are billed. A high `effort` or large `max_tokens` cap can dominate spend on planning-heavy steps.
- **Latency** — More reasoning usually means **longer** first-token and completion times; in a **ReAct** loop, reasoning applies **on each** assistant turn unless you isolate it (e.g. a dedicated planner node only).
- **Quality vs saturation** — Some tasks need deep deliberation once (plan decomposition); others need fast tool iteration. A **staged graph** lets you spend the budget **only where it helps**; a **unified** agent risks paying for reasoning on every hop.
- **Transparency** — You can keep reasoning in the API response for debugging, or set **`exclude: true`** so the model still reasons but you do not receive (or store) the trace—privacy and UI noise trade-off.

OpenRouter’s API allows **`effort`** (OpenAI-style levels: `none` … `xhigh`) **or** **`max_tokens`** (direct cap for Anthropic-style and others)—**use one or the other**, not both, per their docs. Optional **`exclude`** controls whether reasoning appears in the `reasoning` field of the message.

**Note:** Some providers (e.g. parts of the OpenAI o-series) may **not** return visible reasoning text even when reasoning runs internally—see the doc’s note on that.

## What you need

- `OPENROUTER_API_KEY` — [OpenRouter](https://openrouter.ai/keys)
- `EXA_API_KEY` — [Exa](https://exa.ai)
- A **planner** model that supports the `reasoning` options you choose (effort vs `max_tokens` depends on provider—see the guide above).

## Trade-offs (what to look for)

- **Pipeline**: predictable stages, **one** explicit reasoning budget on the planner, easy to log plan vs gather vs write; typically more code.
- **Unified ReAct**: fewer nodes; with reasoning **off**, usually cheaper/faster per turn; with reasoning **on**, cost and latency scale with **number of LLM steps** (each tool cycle).

The run cells print **timers** (pipeline per-node and total wall time for both graphs) so you can compare architectures under the same topic.

## Dependencies

In [ ]:
%pip install -q "langgraph>=0.2" "langchain-openai>=0.3" "langchain-core>=0.3" "openai>=1.40" "python-dotenv>=1.0" "exa_py>=1.0" "pydantic>=2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.4 MB/s eta 0:00:00


## API keys, models, and reasoning budget

Optional environment variables (defaults in code). Reasoning knobs follow [OpenRouter reasoning tokens](https://openrouter.ai/docs/guides/best-practices/reasoning-tokens): use **`effort`** *or* **`max_tokens`**, not both.

| Variable | Role |
|----------|------|
| `OPENROUTER_PLANNER_MODEL` | Planner model (should support your chosen `reasoning` mode) |
| `OPENROUTER_WRITER_MODEL` | Writer for the synthesis step (no reasoning block in this demo) |
| `OPENROUTER_UNIFIED_MODEL` | Single model for the ReAct agent |
| `OPENROUTER_PLANNER_REASONING` | `1` / `true` to send `extra_body.reasoning` on the **planner** (default: on) |
| `OPENROUTER_REASONING_EFFORT` | e.g. `none`, `minimal`, `low`, `medium`, `high`, `xhigh` — used if `OPENROUTER_REASONING_MAX_TOKENS` is **unset** |
| `OPENROUTER_REASONING_MAX_TOKENS` | If set (integer string), planner uses `reasoning.max_tokens` instead of `effort` |
| `OPENROUTER_REASONING_EXCLUDE` | `1` / `true` → `reasoning.exclude: true` (think internally, omit trace from response) |
| `OPENROUTER_UNIFIED_REASONING` | `1` / `true` → attach the **same** `reasoning` policy to **every** unified-agent LLM call (compare cost/latency vs off) |
| `OPENROUTER_UNIFIED_REASONING_EFFORT` | Effort for unified agent when unified reasoning is on (default: `medium`) |

In [ ]:
from __future__ import annotations

import json
import os
import time
from typing import Any

from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "").strip()
EXA_API_KEY = os.getenv("EXA_API_KEY", "").strip()

if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = input("OPENROUTER_API_KEY: ").strip()
if not EXA_API_KEY:
    EXA_API_KEY = input("EXA_API_KEY: ").strip()

OPENROUTER_PLANNER_MODEL = os.getenv("OPENROUTER_PLANNER_MODEL", "openai/o3-mini").strip()
OPENROUTER_WRITER_MODEL = os.getenv("OPENROUTER_WRITER_MODEL", "openai/gpt-4o-mini").strip()
OPENROUTER_UNIFIED_MODEL = os.getenv("OPENROUTER_UNIFIED_MODEL", "openai/gpt-4o-mini").strip()

_pr = os.getenv("OPENROUTER_PLANNER_REASONING", "1").strip().lower()
PLANNER_USE_REASONING = _pr in ("1", "true", "yes", "on")
OPENROUTER_REASONING_EFFORT = os.getenv("OPENROUTER_REASONING_EFFORT", "high").strip()

_mx = os.getenv("OPENROUTER_REASONING_MAX_TOKENS", "").strip()
OPENROUTER_REASONING_MAX_TOKENS: int | None = None
if _mx:
    try:
        OPENROUTER_REASONING_MAX_TOKENS = max(1, int(_mx))
    except ValueError:
        raise ValueError("OPENROUTER_REASONING_MAX_TOKENS must be an integer when set") from None

_ex = os.getenv("OPENROUTER_REASONING_EXCLUDE", "").strip().lower()
OPENROUTER_REASONING_EXCLUDE = _ex in ("1", "true", "yes", "on")

_ur = os.getenv("OPENROUTER_UNIFIED_REASONING", "").strip().lower()
UNIFIED_USE_REASONING = _ur in ("1", "true", "yes", "on")
OPENROUTER_UNIFIED_REASONING_EFFORT = os.getenv("OPENROUTER_UNIFIED_REASONING_EFFORT", "medium").strip()

OR_HEADERS = {
    "HTTP-Referer": os.getenv("OPENROUTER_HTTP_REFERER", "http://localhost"),
    "X-Title": os.getenv("OPENROUTER_APP_TITLE", "LangGraph-DeepResearch-Demo"),
}


def planner_reasoning_payload() -> dict[str, Any]:
    """Build OpenRouter `reasoning` object: max_tokens OR effort (not both)."""
    body: dict[str, Any] = {}
    if OPENROUTER_REASONING_MAX_TOKENS is not None:
        body["max_tokens"] = OPENROUTER_REASONING_MAX_TOKENS
    else:
        body["effort"] = OPENROUTER_REASONING_EFFORT
    if OPENROUTER_REASONING_EXCLUDE:
        body["exclude"] = True
    return body


def unified_reasoning_payload() -> dict[str, Any]:
    """Separate knob so unified agent can use a different effort than planner."""
    return {"effort": OPENROUTER_UNIFIED_REASONING_EFFORT}


print("Planner model:", OPENROUTER_PLANNER_MODEL)
if PLANNER_USE_REASONING:
    print("  Planner reasoning:", planner_reasoning_payload())
else:
    print("  Planner reasoning: off")
print("Writer model:", OPENROUTER_WRITER_MODEL)
print("Unified model:", OPENROUTER_UNIFIED_MODEL, "| unified reasoning:", unified_reasoning_payload() if UNIFIED_USE_REASONING else "off")

Planner model: openai/o3-mini
  Planner reasoning: {'effort': 'high'}
Writer model: openai/gpt-4o-mini
Unified model: openai/gpt-4o-mini | unified reasoning: off


## Imports, schemas, Exa helpers

In [ ]:
from typing import NotRequired, TypedDict

from exa_py import Exa
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import create_react_agent
from openai import OpenAI
from pydantic import BaseModel, Field


class ResearchPlan(BaseModel):
    """Structured plan from the reasoning planner."""

    sub_questions: list[str] = Field(
        ..., min_length=2, max_length=6, description="Concrete sub-questions to answer"
    )
    exa_queries: list[str] = Field(
        ...,
        min_length=2,
        max_length=6,
        description="Distinct Exa search strings (neural search), not copy-paste of each other",
    )
    section_outline: list[str] = Field(
        ..., min_length=3, max_length=8, description="Markdown H2 headings for the final report"
    )


def _strip_json_fence(s: str) -> str:
    s = (s or "").strip()
    if s.startswith("```"):
        lines = s.split("\n")
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        s = "\n".join(lines)
    return s.strip()


def _extract_first_json_object(s: str) -> str:
    s = _strip_json_fence(s)
    start = s.find("{")
    if start == -1:
        return s
    depth = 0
    for i, ch in enumerate(s[start:], start):
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return s[start : i + 1]
    return s


def exa_run_queries(queries: list[str], num_results: int = 4) -> str:
    """Run Exa neural search for each query; return a single string for the writer."""
    exa = Exa(EXA_API_KEY)
    blocks: list[str] = []
    for q in queries:
        q = (q or "").strip()
        if not q:
            continue
        resp = exa.search_and_contents(
            q,
            type="neural",
            num_results=num_results,
            highlights=True,
        )
        blocks.append(f"### Query: {q}\n")
        for idx, r in enumerate(resp.results or [], 1):
            hl = ""
            if getattr(r, "highlights", None):
                hl = " ".join(r.highlights)
            line = (
                f"{idx}. **{getattr(r, 'title', '')}**\n"
                f"   URL: {getattr(r, 'url', '')}\n"
                f"   Snippet: {hl[:1200]}\n"
            )
            blocks.append(line)
        blocks.append("\n")
    return "\n".join(blocks).strip() or "(No Exa results.)"


@tool
def exa_web_search(query: str) -> str:
    """Neural web search via Exa. Pass a focused question or keywords; returns titles, URLs, and highlight snippets."""
    return exa_run_queries([query], num_results=4)


PLANNER_SYSTEM = """You are a research planner for a "deep research" style workflow.
Given one user topic, output ONLY valid JSON (no markdown fences) matching this shape:
{"sub_questions": [...], "exa_queries": [...], "section_outline": [...]}
Rules:
- sub_questions: 3-5 precise questions that together cover the topic.
- exa_queries: 3-5 DISTINCT neural-search queries Exa can run (vary angle: mechanisms, benchmarks, risks, recent news, surveys).
- section_outline: 3-7 H2-style headings for a brief technical memo.
Do not include any text outside the JSON object."""

## Planner call (OpenRouter + optional `reasoning`)

Uses the **OpenAI Python client** pointed at OpenRouter so `extra_body["reasoning"]` is sent reliably (same pattern as the [official Python example](https://openrouter.ai/docs/guides/best-practices/reasoning-tokens)). The **plan** node builds the payload via `planner_reasoning_payload()`:

- **`max_tokens`** if `OPENROUTER_REASONING_MAX_TOKENS` is set, else **`effort`**
- optional **`exclude`** from `OPENROUTER_REASONING_EXCLUDE`

The LangGraph **plan** node times this call and records API **usage** when present.

In [ ]:
def openrouter_openai_client() -> OpenAI:
    return OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
        default_headers=OR_HEADERS,
    )


def plan_with_openrouter(topic: str) -> tuple[ResearchPlan, dict]:
    """Returns validated plan and metadata (e.g. token usage from OpenRouter)."""
    client = openrouter_openai_client()
    req: dict = {
        "model": OPENROUTER_PLANNER_MODEL,
        "temperature": 0.2,
        "messages": [
            {"role": "system", "content": PLANNER_SYSTEM},
            {"role": "user", "content": topic},
        ],
    }
    if PLANNER_USE_REASONING:
        req["extra_body"] = {"reasoning": planner_reasoning_payload()}
    resp = client.chat.completions.create(**req)
    msg = resp.choices[0].message
    raw = (msg.content or "").strip()
    reasoning_dbg = getattr(msg, "reasoning", None) or getattr(msg, "reasoning_content", None)
    if reasoning_dbg and not OPENROUTER_REASONING_EXCLUDE:
        print("--- Planner reasoning trace (provider-dependent; o-series may omit) ---")
        print(str(reasoning_dbg)[:1500] + ("…" if len(str(reasoning_dbg)) > 1500 else ""))
    payload = _extract_first_json_object(raw)
    plan = ResearchPlan.model_validate_json(payload)
    meta: dict = {}
    usage = getattr(resp, "usage", None)
    if usage is not None:
        meta["usage"] = usage.model_dump() if hasattr(usage, "model_dump") else dict(usage)
    return plan, meta


def chat_openrouter(
    model: str,
    temperature: float = 0.2,
    reasoning: dict | None = None,
) -> ChatOpenAI:
    """OpenRouter via LangChain; optional per-request reasoning (e.g. every ReAct step)."""
    kwargs = dict(
        model=model,
        base_url="https://openrouter.ai/api/v1",
        api_key=OPENROUTER_API_KEY,
        default_headers=OR_HEADERS,
        temperature=temperature,
    )
    if reasoning is not None:
        try:
            return ChatOpenAI(**kwargs, extra_body={"reasoning": reasoning})
        except TypeError:
            return ChatOpenAI(**kwargs, model_kwargs={"reasoning": reasoning})
    return ChatOpenAI(**kwargs)

## Architecture A: explicit `StateGraph` (plan → gather → synthesize)

In [ ]:
class PipelineState(TypedDict):
    topic: str
    plan: NotRequired[dict]
    findings: NotRequired[str]
    report: NotRequired[str]
    timings: NotRequired[dict[str, float]]
    planner_meta: NotRequired[dict]


writer_llm = chat_openrouter(OPENROUTER_WRITER_MODEL, temperature=0.3)


def node_plan(state: PipelineState) -> dict:
    t0 = time.perf_counter()
    plan, meta = plan_with_openrouter(state["topic"])
    dt = time.perf_counter() - t0
    tim = dict(state.get("timings") or {})
    tim["plan_llm_s"] = round(dt, 3)
    return {"plan": plan.model_dump(), "timings": tim, "planner_meta": meta}


def node_gather(state: PipelineState) -> dict:
    t0 = time.perf_counter()
    queries = state["plan"].get("exa_queries") or []
    findings = exa_run_queries(queries)
    dt = time.perf_counter() - t0
    tim = dict(state.get("timings") or {})
    tim["gather_exa_s"] = round(dt, 3)
    return {"findings": findings, "timings": tim}


def node_synthesize(state: PipelineState) -> dict:
    t0 = time.perf_counter()
    outline = state["plan"].get("section_outline") or []
    subs = state["plan"].get("sub_questions") or []
    prompt = (
        "Write a concise research memo in Markdown for the topic below.\n"
        "Use ONLY the findings for factual claims; if something is missing, say so.\n"
        "Include inline links using URLs from the findings.\n\n"
        f"# Topic\n{state['topic']}\n\n"
        f"## Planned sub-questions\n{json.dumps(subs, ensure_ascii=False)}\n\n"
        f"## Section outline (follow loosely)\n{json.dumps(outline, ensure_ascii=False)}\n\n"
        f"## Exa findings\n{state['findings']}"
    )
    out = writer_llm.invoke(prompt)
    dt = time.perf_counter() - t0
    tim = dict(state.get("timings") or {})
    tim["synthesize_llm_s"] = round(dt, 3)
    return {"report": out.content, "timings": tim}


g = StateGraph(PipelineState)
g.add_node("plan", node_plan)
g.add_node("gather", node_gather)
g.add_node("synthesize", node_synthesize)
g.add_edge(START, "plan")
g.add_edge("plan", "gather")
g.add_edge("gather", "synthesize")
g.add_edge("synthesize", END)

pipeline_app = g.compile()
try:
    print(pipeline_app.get_graph().draw_mermaid())
except Exception as _exc:
    print("(Mermaid preview skipped)", _exc)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	plan(plan)
	gather(gather)
	synthesize(synthesize)
	__end__([<p>__end__</p>]):::last
	__start__ --> plan;
	gather --> synthesize;
	plan --> gather;
	synthesize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Architecture B: unified ReAct agent (`create_react_agent`)

In [ ]:
UNIFIED_SYSTEM = """You are a careful research assistant (deep-research style).
You must use the tool `exa_web_search` one or more times to gather fresh web evidence, then write the final answer.
Rules:
- Prefer 2-4 focused searches over one vague query.
- After you have enough coverage, STOP calling tools and produce a structured Markdown report:
  short executive summary, sections with bullets, and a **Sources** list with real URLs from the tool output.
- Do not invent URLs.
- If the topic is ambiguous, state assumptions briefly.
"""

_unified_reasoning = unified_reasoning_payload() if UNIFIED_USE_REASONING else None
unified_llm = chat_openrouter(
    OPENROUTER_UNIFIED_MODEL,
    temperature=0.3,
    reasoning=_unified_reasoning,
)
unified_agent = create_react_agent(
    unified_llm,
    tools=[exa_web_search],
    prompt=UNIFIED_SYSTEM,
)

/tmp/ipykernel_511/1316388501.py:17: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  unified_agent = create_react_agent(


## Run both on the same topic (real APIs)

Edit `USER_TOPIC` or set env `DEMO_RESEARCH_TOPIC`. Each block prints **wall-clock seconds**; the pipeline also prints **per-node** times (`plan_llm_s`, `gather_exa_s`, `synthesize_llm_s`) and planner **token usage** when the API returns it.

In [ ]:
USER_TOPIC = os.getenv(
    "DEMO_RESEARCH_TOPIC",
    "What are the main 2024–2025 developments in agentic AI evaluation benchmarks and safety testing?",
)

print("Topic:", USER_TOPIC)

t_pipe_wall0 = time.perf_counter()
pipe_out = pipeline_app.invoke({"topic": USER_TOPIC})
PIPELINE_WALL_S = time.perf_counter() - t_pipe_wall0

timings = pipe_out.get("timings") or {}
print("\n=== Architecture A: staged pipeline ===")
print(f"Wall clock (full graph): {PIPELINE_WALL_S:.3f} s")
print("Per-node (seconds):", json.dumps(timings, indent=2))
usage = (pipe_out.get("planner_meta") or {}).get("usage")
if usage:
    print("Planner token usage (API):", json.dumps(usage, indent=2, default=str))

print("\n--- Plan (excerpt) ---")
print(json.dumps(pipe_out["plan"], indent=2, ensure_ascii=False)[:2000])
print("\n--- Report (excerpt) ---\n")
print(pipe_out["report"][:3500])
if len(pipe_out["report"]) > 3500:
    print("\n… [truncated for display]")

Topic: What are the main 2024–2025 developments in agentic AI evaluation benchmarks and safety testing?

=== Architecture A: staged pipeline ===
Wall clock (full graph): 25.387 s
Per-node (seconds): {
  "plan_llm_s": 10.067,
  "gather_exa_s": 2.487,
  "synthesize_llm_s": 12.819
}
Planner token usage (API): {
  "completion_tokens": 2601,
  "prompt_tokens": 159,
  "total_tokens": 2760,
  "completion_tokens_details": {
    "accepted_prediction_tokens": null,
    "audio_tokens": 0,
    "reasoning_tokens": 1152,
    "rejected_prediction_tokens": null,
    "image_tokens": 0
  },
  "prompt_tokens_details": {
    "audio_tokens": 0,
    "cached_tokens": 0,
    "cache_write_tokens": 0,
    "video_tokens": 0
  },
  "cost": 0.0116193,
  "is_byok": false,
  "cost_details": {
    "upstream_inference_cost": 0.0116193,
    "upstream_inference_prompt_cost": 0.0001749,
    "upstream_inference_completions_cost": 0.0114444
  }
}

--- Plan (excerpt) ---
{
  "sub_questions": [
    "What new evaluation bench

In [ ]:
t_uni_wall0 = time.perf_counter()
agent_out = unified_agent.invoke(
    {"messages": [HumanMessage(content=USER_TOPIC)]},
    config={"recursion_limit": 40},
)
UNIFIED_WALL_S = time.perf_counter() - t_uni_wall0

msgs = agent_out["messages"]
tool_calls = sum(1 for m in msgs if isinstance(m, ToolMessage))
assistant_turns = sum(1 for m in msgs if getattr(m, "type", None) == "ai")
final = msgs[-1].content if msgs else ""

print("\n=== Architecture B: unified ReAct ===")
print(f"Wall clock (full graph): {UNIFIED_WALL_S:.3f} s")
print(f"Assistant messages (LLM steps): {assistant_turns} | Exa tool runs: {tool_calls}")
if UNIFIED_USE_REASONING:
    print("(Unified reasoning is ON — each assistant step may use your reasoning budget; compare wall time to planner-only reasoning.)")

print("\n--- Final answer (excerpt) ---\n")
print(final[:4500])
if len(final) > 4500:
    print("\n… [truncated for display]")


=== Architecture B: unified ReAct ===
Wall clock (full graph): 15.725 s
Assistant messages (LLM steps): 2 | Exa tool runs: 2

--- Final answer (excerpt) ---

# Developments in Agentic AI Evaluation Benchmarks and Safety Testing (2024-2025)

## Executive Summary
The landscape of agentic AI evaluation benchmarks and safety testing is rapidly evolving as we approach 2024-2025. Key developments include the introduction of new benchmarks designed for dynamic environments, collaborative safety evaluations, and enhanced safety testing methodologies. These advancements aim to address the complexities of evaluating AI agents and ensuring their safe deployment in real-world applications.

## Key Developments in Agentic AI Evaluation Benchmarks

- **Gaia2 Benchmark**:
  - Introduced for evaluating large language model (LLM) agents in dynamic and asynchronous environments.
  - Focuses on realistic scenarios where time, uncertainty, and collaboration are critical.
  - Aims to overcome limitations 

### Timing comparison

Run after the pipeline and unified cells so `PIPELINE_WALL_S` and `UNIFIED_WALL_S` exist.

In [ ]:
try:
    print("=== Timing comparison (same topic) ===")
    print(f"Architecture A (pipeline) wall clock: {PIPELINE_WALL_S:8.3f} s")
    print(f"Architecture B (unified)  wall clock: {UNIFIED_WALL_S:8.3f} s")
    pt = pipe_out.get("timings") or {}
    if pt:
        inner = " + ".join(f"{k}={v}" for k, v in sorted(pt.items()))
        print(f"  Pipeline breakdown: {inner}")
except NameError:
    print("Run the pipeline cell, then the unified cell, then re-run this box.")

=== Timing comparison (same topic) ===
Architecture A (pipeline) wall clock:   25.387 s
Architecture B (unified)  wall clock:   15.725 s
  Pipeline breakdown: gather_exa_s=2.487 + plan_llm_s=10.067 + synthesize_llm_s=12.819


## Takeaways

- **Pipeline** spends the OpenRouter **reasoning budget once** on the planner (if enabled), then uses a cheap writer—good when planning quality matters more than iterative tool deliberation. Tweak **`effort`** vs **`max_tokens`** (see [reasoning tokens](https://openrouter.ai/docs/guides/best-practices/reasoning-tokens)) to trade quality, latency, and billed output tokens.
- **Unified ReAct** is quicker to author; with reasoning **off**, it is often faster per run; with **`OPENROUTER_UNIFIED_REASONING`**, reasoning may apply **every** assistant step—wall time and token use often jump sharply. That illustrates **why** graph design and budget placement matter.
- Use the printed **timers** and planner **usage** to compare runs at the same `USER_TOPIC`. For fair comparisons, align search breadth (pipeline uses all planned Exa queries; ReAct chooses its own count).

### Experiments to try

- Set `OPENROUTER_REASONING_EFFORT=low` then `high` on the planner only; compare `plan_llm_s` and memo quality.
- Set `OPENROUTER_REASONING_MAX_TOKENS=1500` (clear `effort` by using max_tokens path) for Anthropic-style models.
- Set `OPENROUTER_REASONING_EXCLUDE=true` to hide reasoning text while still consuming reasoning capacity (per OpenRouter docs).
- Toggle `OPENROUTER_UNIFIED_REASONING` and watch **Architecture B** wall time vs **Architecture A**.